# 02 Baseline

## import data and cv

In [2]:
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from src.preprocessing import RANDOM_STATE, load_raw, split_xy, load_saved_split, make_preprocessor

df = load_raw()
X, y = split_xy(df)
X_train, X_test, y_train, y_test = load_saved_split(X, y)
print("train:", X_train.shape, f"離職比例 {y_train.mean():.3f}")

train: (1176, 30) 離職比例 0.162


eda: 把每筆資料固定分配到一次 train 或 test 的 單次 80/20 train/test split

CV: cross-validation (fold), 5 fold *3

In [3]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
splits = list(cv.split(X_train, y_train))
print("total", len(splits), " fold")

total 15  fold


## Evaluate

ROC-AUC 看模型能不能把離職者排在留任者前面

PR-AUC 對不平衡資料更敏感，亂猜的基準大約等於離職比例 0.16，而不是 0.5

In [5]:
def cv_eval(name, kind, model):
    """kind 是 'linear' 或 'tree'，決定用哪一種預處理"""
    rows = []
    for i, (tr, va) in enumerate(splits):
        Xtr, Xva = X_train.iloc[tr], X_train.iloc[va]
        ytr, yva = y_train.iloc[tr], y_train.iloc[va]

        prep = make_preprocessor(kind).fit(Xtr, ytr)      # 只用訓練折 fit
        m = clone(model).fit(prep.transform(Xtr), ytr)
        p = m.predict_proba(prep.transform(Xva))[:, 1]    # 驗證折的離職機率

        rows.append({"model": name, "fold": i,
                     "roc_auc": roc_auc_score(yva, p),
                     "pr_auc": average_precision_score(yva, p)})
    return pd.DataFrame(rows)

### dummy

In [6]:
from sklearn.dummy import DummyClassifier

results = []
results.append(cv_eval("dummy", "linear", DummyClassifier(strategy="prior")))
results[-1].groupby("model")[["roc_auc", "pr_auc"]].mean().round(3)

,roc_auc,pr_auc
model,,
dummy,0.5,0.162


### model

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

pos_w = (1 - y_train.mean()) / y_train.mean()

models = {
    "logreg": ("linear", LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000)),
    "rf":     ("tree",   RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                 random_state=RANDOM_STATE, n_jobs=-1)),
    "lgbm":   ("tree",   LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15,
                                         scale_pos_weight=pos_w, random_state=RANDOM_STATE, verbose=-1)),
}

for name, (kind, model) in models.items():
    results.append(cv_eval(name, kind, model))
    print(name, "done")

logreg done
rf done
lgbm done


還沒調參

In [9]:
res = pd.concat(results, ignore_index=True)
summary = res.groupby("model")[["roc_auc", "pr_auc"]].agg(["mean", "std"]).round(3)
display(summary.sort_values(("pr_auc", "mean"), ascending=False))

Path("reports").mkdir(exist_ok=True)
res.to_csv("reports/cv_baseline_folds.csv", index=False)
summary.to_csv("reports/cv_baseline_summary.csv")

roc_auc        pr_auc       
          mean    std   mean    std
model                              
logreg   0.831  0.041  0.629  0.063
lgbm     0.799  0.037  0.552  0.079
rf       0.800  0.036  0.552  0.072
dummy    0.500  0.000  0.162  0.000